In [2]:
import os
import pandas as pd
import re
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [3]:
def load_data():
    """Kết nối DB"""
    load_dotenv()
    
    try:
        connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
        engine = create_engine(connection_string)
        print("Kết nối CSDL PostgreSQL thành công!")
    except Exception as e:
        print(f"LỖI: Không thể kết nối CSDL. Lỗi: {e}")
        return None

    # Các câu lệnh SQL
    sql_movies = "SELECT id, title, summary as description FROM Movies"
    sql_genres = "SELECT movie_id, STRING_AGG(G.name, ', ') AS genres FROM Movie_Genres MG JOIN Genres G ON MG.genre_id = G.id GROUP BY MG.movie_id"
    sql_actors = "SELECT movie_id, STRING_AGG(A.name, ', ') AS actors FROM Movie_Actors MA JOIN Actors A ON MA.actor_id = A.id GROUP BY MA.movie_id"
    sql_directors = "SELECT movie_id, STRING_AGG(D.name, ', ') AS directors FROM Movie_Directors MD JOIN Directors D ON MD.director_id = D.id GROUP BY MD.movie_id"

    all_genres = "SELECT * FROM genres"

    try:
        print("Đang tải dữ liệu (movies, genres, actors, directors)...")
        df_movies = pd.read_sql(sql_movies, engine)
        df_genres = pd.read_sql(sql_genres, engine)
        df_actors = pd.read_sql(sql_actors, engine)
        df_directors = pd.read_sql(sql_directors, engine)


        # Gộp (Merge) tất cả
        df_full = pd.merge(df_movies, df_genres, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_actors, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_directors, left_on='id', right_on='movie_id', how='left')
        
        df_full = df_full.drop(columns=['movie_id_x', 'movie_id_y', 'movie_id'])
        print("Tải và gộp dữ liệu thành công!")

        # Tải tất cả genres
        df_all_genres = pd.read_sql(all_genres, engine)
        print("Tải tất cả genres thành công!")
        return df_full, df_all_genres

    except Exception as e:
        print(f"LỖI: Không thể tải dữ liệu. Lỗi: {e}")
        return None

In [4]:
def clean_text(text):
    """Hàm dọn dẹp (xóa dấu cách, viết thường)"""
    if text is None:
        return ""
    names = [re.sub(r'\s+', '', name).lower() for name in text.split(', ')]
    return " ".join(names)

In [5]:
def create_soup(data_row):
    """Hàm trộn các nguyên liệu (genres, actors, directors)"""
    genres = clean_text(data_row['genres'])
    actors = clean_text(data_row['actors'])
    directors = clean_text(data_row['directors'])
    description = clean_text(data_row['description'])
    return f"{genres} {actors} {directors} {description}"

In [11]:
# Load data from database
df_movies, df_genres = load_data()
df_movies.head()



Kết nối CSDL PostgreSQL thành công!
Đang tải dữ liệu (movies, genres, actors, directors)...
Tải và gộp dữ liệu thành công!
Tải tất cả genres thành công!


,id,title,description,genres,actors,directors
0,37702,Forbidden City Cop,An imperial agent gets ridiculed for his vario...,"Adventure, Action, Comedy","Stephen Chow, Carina Lau, Carman Lee Yeuk-Tung...",Vincent Kok Tak-Chiu
1,1290821,Shelter,A man living in self-imposed exile on a remote...,"Action, Thriller, Crime","Jason Statham, Bodhi Rae Breathnach, Bill Nigh...",Ric Roman Waugh
2,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"Thriller, Crime","Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",Bart Layton
3,1198994,Send Help,Two colleagues become stranded on a deserted i...,"Horror, Comedy, Thriller","Rachel McAdams, Dylan O'Brien, Edyll Ismail, D...",Sam Raimi
4,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"Adventure, Thriller, Science Fiction","Gerard Butler, Morena Baccarin, Roman Griffin ...",Ric Roman Waugh


In [12]:
df_movies.to_csv('movies_data.csv', index=False)

In [13]:
df_genres

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime
5,99,Documentary
6,18,Drama
7,10751,Family
8,14,Fantasy
9,36,History


# 5 nhóm personal
1. Adrenaline: (28)Action, (12)Adventure, (53)Thriller, (10752)War, (37)Western, (878)Science Fiction
2. Dreamy: 16(Animation), 10751(Family), 14(Fantasy), 10402(Music), 35(Comedy)     
3. Dark: 27(Horror), 80(Crime), 9648(Mystery), 53(Thriller)                
4. Emotional: 18(Drama), 10749(Romance), 36(History), 10770(TV Movie), 35(Comedy)
5. Geek: 878(Science Fiction), 99(Documentary), 36(History)

# 3 kiểu user
1. Expressive: Nhóm nhiệt tình: data sẽ có cả like, rating, watch (Chiếm 30%)
2. Passive: Nhóm bị động: Có xem nhưng lười chấm điểm (Chiếm 50%)
3. Forgetful: Nhóm tâm trạng tùy hứng (chiếm 20%) 

In [17]:
from datetime import datetime, timedelta
import random

# Định nghĩa các Cụm sở thích (Personas) dựa trên thể loại phim
personas = {
    'adrenaline': ["Action", "Adventure", "Thriller", "War", "Western", "Science Fiction"],
    'dreamy': ["Animation", "Family", "Fantasy", "Music", "Comedy"],
    'dark': ["Horror", "Crime", "Mystery", "Thriller"],
    'emotional': ["Drama", "Romance", "History", "TV Movie", "Comedy"],
    'geek': ["Science Fiction", "Documentary", "Comedy"]
    
}

In [18]:
def generate_interaction(df_movies, num_users = 500):
    data = []
    for uid in range(10, num_users + 1):
        # Gán cá tính hành vi cho user
        behavior = random.choices(['expressive', 'passive', 'forgetful'], weights=[0.3, 0.5, 0.2])[0]


        # Gán một persona ngẫu nhiên
        p_name = random.choice(list(personas.keys()))
        fav_genres = personas[p_name]

        # Sinh 40 - 70 lượt tương tác cho mỗi user
        for _ in range(random.randint(40, 70)):
            if behavior == 'forgetful':
                is_match = random.random() < 0.3 # Nhóm vãng lai chỉ xem đúng gu 30%
            else:
                is_match = random.random() < 0.8 # Nhóm Tích cực/Thụ động xem đúng gu 80%

            if is_match:
                pattern = '|'.join(fav_genres) # Toán tử OR cho regex
                target_movies = df_movies[df_movies['genres'].str.contains(pattern)]
            else:
                target_movies = df_movies

            movie = target_movies.sample(n=1).iloc[0]

            # Logic sinh hệ số 
            if is_match:
                if behavior == 'forgetful':
                    progress = random.randint(30, 80)
                    favourite = 0 
                    rating = random.randint(2, 5) if random.random() < 0.3 else None # Rate lung tung
                
                # Nhóm thụ động (passive)
                elif behavior == 'passive':
                    progress = random.randint(80, 100)
                    favourite = 0
                    rating = None
                
                # Nhóm tích cực (expressive)
                else:
                    progress = random.randint(70, 100)
                    favourite = 1 if random.random() < 0.7 else 0 # Tỉ lệ like cao
                    rating = random.randint(4, 5)
            else:
                progress = random.randint(5, 50)
                favourite = 0
                rating = random.randint(1, 3) if random.random() < 0.1 else None

            data.append({
                'user_id': uid,
                'movie_id': movie['id'],
                'rating': rating,
                'max_progress_percent': progress,
                'favourite': favourite, 
                'in_watchlist': 1 if random.random() < 0.2 else 0, 
                'view_count': random.randint(1, 8) if is_match else 1,
                'last_watched_at': datetime.now() - timedelta(days=random.randint(0, 180))
            })
    return pd.DataFrame(data)


In [19]:
df_interactions = generate_interaction(df_movies)

In [20]:
df_interactions

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748
...,...,...,...,...,...,...,...,...
27011,500,15276,5.0,97,1,0,4,2025-12-06 01:59:01.806758
27012,500,736545,5.0,81,1,1,3,2026-03-08 01:59:01.818289
27013,500,11297,4.0,92,0,0,2,2025-10-01 01:59:01.826666
27014,500,146724,5.0,90,1,0,3,2025-10-04 01:59:01.838193


In [16]:
df_movies.iloc[df_movies['id'] == 915295]


,id,title,description,genres,actors,directors
10510,915295,Your Turn to Kill: The Movie,Newlyweds Nana and Shota move into an apartmen...,"Thriller, Mystery","Tomoyo Harada, Kei Tanaka, Nanase Nishino, Ryu...",Noriyoshi Sakuma


In [21]:
df_interactions.to_csv('interactions_data.csv', index=False)